# Sentiment Analysis

The canoncial NLP task, Most of what you need to know about classical text classification shows up here.

## Problem Definition

"The food was not great."  Positive or Negative ?

## Basic Concept

Classical sentiment is a two-step recipe.

* Represent. Turn the text info a feature vector.
* Classify. Fit a linear model on labeled examples.

Naive bayes way. Assume every feature is independent given the lable.
Estimate P(word | positive) and P(word | negative) from counts. At interence, multiply the probabilities.

Logistic regression way. It learns a weight per feature, including nagative weights.

# Build your Own

## Mini-Dataset

In [ ]:
POSITIVE = [
    "absolutely loved this movie",
    "beautiful cinematography and a great story",
    "one of the best films of the year",
    "brilliant acting from the lead",
    "heartwarming and funny",
]

NEGATIVE = [
    "boring and far too long",
    "not worth your time",
    "the plot made no sense",
    "terrible acting, awful script",
    "i want my two hours back",
]


## Naive Bayes Way

In [7]:
import math
from collections import Counter

def train_nb(docs_by_class, vocab, alpha=1.0):
    class_priors = {}
    class_word_probs = {}
    total_docs = sum(len(d) for d in docs_by_class.values())

    for cls, docs in docs_by_class.items():
        class_priors[cls] = len(docs) / total_docs
        counts = Counter()
        for doc in docs:
            for token in doc:
                counts[token] += 1
        # Smoothing
        total = sum(counts.values()) + alpha * len(vocab)
        class_word_probs[cls] = {
            w: (counts[w] + alpha) / total for w in vocab
        }

    return class_priors, class_word_probs

def predict_nb(text, class_priors, class_word_probs):
    scores = {}
    for cls in class_priors:
        scores[cls] = math.log(class_priors[cls])
        for token in text:
            if token in class_word_probs[cls]:
                scores[cls] += math.log(class_word_probs[cls][token])
    return max(scores, key=scores.get)

docs_by_class = {
    "positive": POSITIVE,
    "negative": NEGATIVE,
}

vocab = set(word for docs in docs_by_class.values() for doc in docs for word in doc.split())
print(vocab)

class_priors, class_word_probs = train_nb(docs_by_class, vocab)
print(class_priors)
print(class_word_probs)

predict_nb("absolutely love this movie", class_priors, class_word_probs)


{'far', 'funny', 'time', 'beautiful', 'terrible', 'acting,', 'loved', 'want', 'this', 'brilliant', 'not', 'i', 'my', 'made', 'acting', 'your', 'from', 'script', 'heartwarming', 'the', 'and', 'boring', 'worth', 'lead', 'great', 'too', 'plot', 'no', 'of', 'movie', 'two', 'absolutely', 'a', 'awful', 'films', 'sense', 'best', 'long', 'year', 'back', 'cinematography', 'story', 'one', 'hours'}
{'positive': 0.5, 'negative': 0.5}
{'positive': {'far': 0.005050505050505051, 'funny': 0.005050505050505051, 'time': 0.005050505050505051, 'beautiful': 0.005050505050505051, 'terrible': 0.005050505050505051, 'acting,': 0.005050505050505051, 'loved': 0.005050505050505051, 'want': 0.005050505050505051, 'this': 0.005050505050505051, 'brilliant': 0.005050505050505051, 'not': 0.005050505050505051, 'i': 0.050505050505050504, 'my': 0.005050505050505051, 'made': 0.005050505050505051, 'acting': 0.005050505050505051, 'your': 0.005050505050505051, 'from': 0.005050505050505051, 'script': 0.005050505050505051, 'hea

'positive'

## Logistic Regression

In [ ]:
from math import log
import numpy as np

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))

def train_lr(X, y, epochs=500, lr=0.05, l2=0.01):
    n_features = X.shape[1]
    w = np.zeros(n_features)
    b = 0.0

    for _ in range(epochs):
        logits = X @ w + b
        preds = sigmoid(logits)
        err = preds - y
        grad_w = X.T @ err / len(y) + l2 * w
        grad_b = err.mean()

        w -= lr * grad_w
        b -= lr * grad_b

    return w, b

def predict_lr(X, w, b):
    return (sigmoid(X @ w + b) >= 0.5).astype(int)
    